In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
import pickle
import warnings
warnings.filterwarnings('ignore')

 # STEP 1 : Data Cleaning and Preprocessing

In [14]:
df = pd.read_csv('../data/combined_dns_data.csv')
print(f'\nInitial shape: {df.shape}')
print(f'Label distribution:\n{df["Label"].value_counts()}')


Initial shape: (1167269, 36)
Label distribution:
Label
0    917300
1    249969
Name: count, dtype: int64


In [15]:
# Remove duplicates
initial_rows = df.shape[0]
df = df.drop_duplicates()
duplicate_rows = initial_rows - df.shape[0]
print(f'\nRemoved duplicates: {duplicate_rows} rows')


Removed duplicates: 0 rows


In [16]:
initial_rows = df.shape[0]
missing_counts = df.isnull().sum()
if missing_counts.sum() > 0:
    df = df.dropna()
    print(f'Removed missing values: {initial_rows - df.shape[0]} rows')
else:
    print(' No missing values')

Removed missing values: 8028 rows


## STEP 3: OUTLIER & SKEW TREATMENT (after train/test split, NO DATA LEAKAGE)

In [18]:
# Split features and target
X = df.drop('Label', axis=1)   
y = df['Label']
print(f'\nFeatures shape: {X.shape}, Target shape: {y.shape}')

print('\n' + '=' * 60)
print('STEP 2: TRAIN / VAL / TEST SPLIT (70/15/15)')
print('=' * 60)

# First split: 70% train, 30% temp (val+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Second split: Split temp into 50% val, 50% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f'\nTrain: {X_train.shape[0]} rows ({100*len(X_train)/len(X):.0f}%)')
print(f'Val:   {X_val.shape[0]} rows ({100*len(X_val)/len(X):.0f}%)')
print(f'Test:  {X_test.shape[0]} rows ({100*len(X_test)/len(X):.0f}%)')

print(f'\nTrain class balance: {y_train.value_counts()[0]} benign, {y_train.value_counts()[1]} malicious')
print(f'Val class balance:   {y_val.value_counts()[0]} benign, {y_val.value_counts()[1]} malicious')
print(f'Test class balance:  {y_test.value_counts()[0]} benign, {y_test.value_counts()[1]} malicious')


Features shape: (1159241, 35), Target shape: (1159241,)

STEP 2: TRAIN / VAL / TEST SPLIT (70/15/15)

Train: 811468 rows (70%)
Val:   173886 rows (15%)
Test:  173887 rows (15%)

Train class balance: 636688 benign, 174780 malicious
Val class balance:   136433 benign, 37453 malicious
Test class balance:  136434 benign, 37453 malicious


In [19]:
#check type of features 
print(f"Feature types:\n{X_train.dtypes}")

Feature types:
SourceIP                                   object
DestinationIP                              object
SourcePort                                  int64
DestinationPort                             int64
TimeStamp                                  object
Duration                                  float64
FlowBytesSent                               int64
FlowSentRate                              float64
FlowBytesReceived                           int64
FlowReceivedRate                          float64
PacketLengthVariance                      float64
PacketLengthStandardDeviation             float64
PacketLengthMean                          float64
PacketLengthMedian                        float64
PacketLengthMode                            int64
PacketLengthSkewFromMedian                float64
PacketLengthSkewFromMode                  float64
PacketLengthCoefficientofVariation        float64
PacketTimeVariance                        float64
PacketTimeStandardDeviation        

In [20]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Store clipping bounds and skew flags computed from TRAIN only
clipping_bounds = {}
log1p_features = []

for col in numeric_cols:
    q1 = X_train[col].quantile(0.01)
    q99 = X_train[col].quantile(0.99)
    clipping_bounds[col] = (q1, q99)

for col in numeric_cols:
    if X_train[col].min() >= 0:
        skewness = X_train[col].skew()
        if abs(skewness) > 1.0:
            log1p_features.append(col)

In [21]:
# Apply clipping bounds (computed from train only) to all sets
for col in numeric_cols:
    q1, q99 = clipping_bounds[col]
    X_train[col] = X_train[col].clip(lower=q1, upper=q99)
    X_val[col] = X_val[col].clip(lower=q1, upper=q99)
    X_test[col] = X_test[col].clip(lower=q1, upper=q99)

# Apply log1p transform (using features identified from train only)
for col in log1p_features:
    X_train[col] = np.log1p(X_train[col])
    X_val[col] = np.log1p(X_val[col])
    X_test[col] = np.log1p(X_test[col])

print(f'Applied percentile clipping and log1p to {len(numeric_cols)} features')
print(f'  - Clipping bounds computed from training set only')
print(f'  - Applied to train, val, and test sets with same bounds')

Applied percentile clipping and log1p to 31 features
  - Clipping bounds computed from training set only
  - Applied to train, val, and test sets with same bounds


# STEP 4: FEATURE SELECTION & REDUCTION (computed on train only, NO DATA LEAKAGE)

In [22]:
# Select numeric features
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
X_train_numeric = X_train[numeric_features]
X_val_numeric = X_val[numeric_features]
X_test_numeric = X_test[numeric_features]

# Drop high-correlation duplicates (based on train only)
corr_matrix = X_train_numeric.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
X_train_numeric = X_train_numeric.drop(columns=to_drop)
X_val_numeric = X_val_numeric.drop(columns=to_drop)
X_test_numeric = X_test_numeric.drop(columns=to_drop)

print(f'Dropped {len(to_drop)} high-corr features')
print(f'Features after dropping: {X_train_numeric.shape[1]}')

Dropped 6 high-corr features
Features after dropping: 25


In [23]:
# Keep top MI (Mutual Information) features (based on train only)
mi_scores = mutual_info_classif(X_train_numeric, y_train, random_state=42)
mi_df = pd.DataFrame({
    'feature': X_train_numeric.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False) 
print(f'\nTop 10 features by MI score (computed from training set only):\n{mi_df.head(10)}')


Top 10 features by MI score (computed from training set only):
                               feature  mi_score
9                     PacketLengthMode  0.554956
8                   PacketLengthMedian  0.478110
2                        FlowBytesSent  0.406950
7                     PacketLengthMean  0.400493
6                 PacketLengthVariance  0.386186
4                    FlowBytesReceived  0.350516
10          PacketLengthSkewFromMedian  0.332964
12  PacketLengthCoefficientofVariation  0.327668
1                             Duration  0.312557
11            PacketLengthSkewFromMode  0.305869


In [24]:
# Keep top MI features (computed from train only, applied to all sets)
top_k = min(20, len(mi_df))
top_features = mi_df.head(top_k)['feature'].tolist()
print(f'\nSelected top {top_k} MI features:\n{top_features}')
X_train = X_train_numeric[top_features]
X_val = X_val_numeric[top_features]
X_test = X_test_numeric[top_features]

print(f'Selected top {top_k} MI features from {len(mi_df)} total')
print(f'Final feature set: {X_train.shape[1]} features')
print(f'  - MI scores computed from training set only')
print(f'  - Same features selected for val and test')


Selected top 20 MI features:
['PacketLengthMode', 'PacketLengthMedian', 'FlowBytesSent', 'PacketLengthMean', 'PacketLengthVariance', 'FlowBytesReceived', 'PacketLengthSkewFromMedian', 'PacketLengthCoefficientofVariation', 'Duration', 'PacketLengthSkewFromMode', 'PacketTimeMedian', 'ResponseTimeTimeMedian', 'FlowReceivedRate', 'PacketTimeSkewFromMedian', 'PacketTimeCoefficientofVariation', 'FlowSentRate', 'ResponseTimeTimeSkewFromMode', 'PacketTimeSkewFromMode', 'ResponseTimeTimeMean', 'ResponseTimeTimeSkewFromMedian']
Selected top 20 MI features from 25 total
Final feature set: 20 features
  - MI scores computed from training set only
  - Same features selected for val and test


In [28]:
print('\n' + '=' * 60)
print('STEP 5: FEATURE SCALING (RobustScaler)')
print('=' * 60)

# RobustScaler uses IQR (Interquartile Range) - resistant to outliers
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_train.columns)

print(f'Scaled train - median: {X_train_scaled.values.mean():.4f}, IQR preserved')
print('RobustScaler fit on train set only (no data leakage)')

#save the scaler for later use
scaler_path = Path('../models/robust_scaler.pkl')
scaler_path.parent.mkdir(parents=True, exist_ok=True)
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'Saved RobustScaler to {scaler_path}')


STEP 5: FEATURE SCALING (RobustScaler)
Scaled train - median: 0.3320, IQR preserved
RobustScaler fit on train set only (no data leakage)
Saved RobustScaler to ..\models\robust_scaler.pkl


In [30]:
print('\n' + '=' * 60)
print('STEP 6: CLASS IMBALANCE HANDLING (SMOTE)')
print('=' * 60)

print('\nBefore SMOTE:')
print(f'  Benign: {(y_train == 0).sum():,}')
print(f'  Malicious: {(y_train == 1).sum():,}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print('\nAfter SMOTE:')
print(f'  Benign: {(y_train_balanced == 0).sum():,}')
print(f'  Malicious: {(y_train_balanced == 1).sum():,}')

X_train_balanced = pd.DataFrame(X_train_balanced, columns=X_train.columns)


STEP 6: CLASS IMBALANCE HANDLING (SMOTE)

Before SMOTE:
  Benign: 636,688
  Malicious: 174,780

After SMOTE:
  Benign: 636,688
  Malicious: 636,688


In [ ]:
print('\n' + '=' * 60)
print('PREPROCESSING VERIFICATION')
print('=' * 60)

print(f'\nTrain (SMOTE balanced): {X_train_balanced.shape}')
print(f'Val (original): {X_val_scaled.shape}')
print(f'Test (original): {X_test_scaled.shape}')
print(f'\nNull values in train: {X_train_balanced.isnull().sum().sum()}')
print(f'Null values in val: {X_val_scaled.isnull().sum().sum()}')
print(f'Null values in test: {X_test_scaled.isnull().sum().sum()}')



PREPROCESSING VERIFICATION

Train (SMOTE balanced): (1273376, 20)
Val (original): (173886, 20)
Test (original): (173887, 20)

Null values in train: 0
Null values in val: 0
Null values in test: 0


In [32]:
print('\n' + '=' * 60)
print('STEP 7: SAVING PREPROCESSED DATA')
print('=' * 60)

train_df = X_train_balanced.copy()
train_df['Label'] = y_train_balanced.values

val_df = X_val_scaled.copy()
val_df['Label'] = y_val.values

test_df = X_test_scaled.copy()
test_df['Label'] = y_test.values

train_output = Path('../data/train_data.csv')
val_output = Path('../data/val_data.csv')
test_output = Path('../data/test_data.csv')

train_df.to_csv(train_output, index=False)
val_df.to_csv(val_output, index=False)
test_df.to_csv(test_output, index=False)

print(f'✓ Train data: {train_df.shape[0]:,} rows → {train_output.name}')
print(f'✓ Val data:   {val_df.shape[0]:,} rows → {val_output.name}')
print(f'✓ Test data:  {test_df.shape[0]:,} rows → {test_output.name}')


STEP 7: SAVING PREPROCESSED DATA
✓ Train data: 1,273,376 rows → train_data.csv
✓ Val data:   173,886 rows → val_data.csv
✓ Test data:  173,887 rows → test_data.csv


In [34]:
print('\n' + '=' * 60)
print('STEP 8: SAVING PREPROCESSING METADATA')
print('=' * 60)

# Save scaler
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✓ RobustScaler saved')

# Save feature names
with open('../data/feature_names.txt', 'w') as f:
    f.write('\n'.join(X_train.columns.tolist()))
print(f'✓ Feature names saved ({len(X_train.columns)} features)')

# Save preprocessing config
config_text = f"""Preprocessing Configuration
===========================
✓ NO DATA LEAKAGE - Steps 2-6 executed with proper train/val/test separation

STEP 1 - Data Cleaning: Removed duplicates, NaN, inf values
STEP 2 - Train/Val/Test Split: 70% / 15% / 15% (stratified)
STEP 3 - Outlier Treatment: 1st/99th percentile clipping (based on train)
         Log1p transform for highly skewed features (based on train)
STEP 4 - Feature Selection: Dropped high-corr (>0.95, based on train)
         Kept top 20 MI features (computed from train only)
STEP 5 - Feature Scaling: RobustScaler (IQR-based, fit on train only)
STEP 6 - Class Balancing: SMOTE on training set only

Final Dataset:
  - Train (balanced): {len(train_df):,} rows
  - Val (original): {len(val_df):,} rows
  - Test (original): {len(test_df):,} rows
  - Features: {X_train.shape[1]}
"""

with open('../data/preprocessing_config.txt', 'w') as f:
    f.write(config_text)
print('✓ Preprocessing config saved')

print(f'\n✓ Ready for model training ({X_train.shape[1]} features)')
print('✓ All preprocessing steps are FREE of data leakage')


STEP 8: SAVING PREPROCESSING METADATA
✓ RobustScaler saved
✓ Feature names saved (20 features)
✓ Preprocessing config saved

✓ Ready for model training (20 features)
✓ All preprocessing steps are FREE of data leakage
